# 04. Neural Network Classification with Biological Data

## Learning objectives

In this notebook, we will build a multiclass classification model with **Keras 3** using a real biological dataset. By the end, you should be able to:

- Explain the structure of a multiclass classification problem.
- Load and inspect a biological dataset from the UCI Machine Learning Repository.
- Encode categorical target labels for a neural network.
- Split data into training, validation, and test sets without leakage.
- Build and train a feedforward neural network with a softmax output layer.
- Evaluate a multiclass model using more than accuracy.
- Recognize the effects of class imbalance and discuss model limitations.

> **Scientific scope:** This is an educational machine-learning exercise. The dataset contains protein-related features and cellular-localization labels. The model is not a validated biological tool and its predictions should not be interpreted as experimental evidence.

## 1. Problem definition

The task is to predict the cellular localization site associated with a protein using a set of numerical features.

For an input sample with feature vector

$$
\mathbf{x} = [x_1, x_2, \ldots, x_8]
$$

the model estimates a probability for each of the $K=10$ classes:

$$
\hat{\mathbf{p}} =
[\hat{p}_1,\hat{p}_2,\ldots,\hat{p}_{10}]
$$

The predicted class is the class with the largest estimated probability:

$$
\hat{y} = \operatorname{argmax}_{k} \hat{p}_k
$$

### Dataset overview

| Property | Description |
|---|---|
| Dataset | Yeast |
| Repository | UCI Machine Learning Repository |
| Samples | 1,484 |
| Input features | 8 numerical features |
| Target | Protein cellular-localization site |
| Number of classes | 10 |
| Missing values | None reported by UCI |
| License | CC BY 4.0 |

The dataset is small by modern deep-learning standards. This makes it useful for learning the workflow, but it also means that a neural network may not outperform simpler machine-learning methods.

**Source:** Nakai, K. (1991). *Yeast* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5KG68

## 2. Environment and imports

We use Keras 3 with the TensorFlow backend in this learning path. Keras is the high-level modeling API; TensorFlow is the selected backend.

The notebook is designed to run on a CPU. Native Windows installations of TensorFlow 2.11 and later do not provide official CUDA GPU support, so a compatible NVIDIA GPU setup is not required here.

In [ ]:
# Select the TensorFlow backend before importing Keras.
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import keras
from keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
keras.utils.set_random_seed(RANDOM_STATE)

print("Python:", sys.version.split()[0])
print("Keras:", keras.__version__)
print("Backend:", keras.backend.backend())

## 3. Download and load the dataset

The `ucimlrepo` package provides a convenient interface to UCI datasets. Install it if it is not already available.

The first download requires an internet connection. If you prefer to work offline, download the dataset files from the UCI page and adapt this section to read the local file.

In [ ]:
# Install the UCI client if necessary.
# In Jupyter, uncomment the next line if the package is missing:
# %pip install ucimlrepo

from ucimlrepo import fetch_ucirepo

# Fetch UCI dataset ID 110 (Yeast).
yeast = fetch_ucirepo(id=110)

# Separate the input features and target.
X_raw = yeast.data.features.copy()
y_raw = yeast.data.targets.copy()

print("Feature shape:", X_raw.shape)
print("Target shape:", y_raw.shape)

display(X_raw.head())
display(y_raw.head())

### 3.1 Inspect dataset metadata

Never assume that a downloaded dataset has the structure you expect. Inspect its columns, data types, missing values, and target labels before modeling.

In [ ]:
# Inspect feature names, data types, and missing values.
print("Feature columns:")
print(X_raw.columns.tolist())

print("\nFeature data types:")
print(X_raw.dtypes)

print("\nMissing values in features:")
print(X_raw.isna().sum())

print("\nTarget columns:")
print(y_raw.columns.tolist())

print("\nMissing values in target:")
print(y_raw.isna().sum())

print("\nUCI metadata:")
print(yeast.metadata)

### 3.2 Prepare a one-dimensional target array

The UCI client returns the target as a DataFrame. We convert it into a one-dimensional Series and inspect the class distribution.

Class imbalance matters because a model can achieve a seemingly reasonable accuracy by predicting common classes while performing poorly on rare classes.

In [ ]:
# The dataset has one target column: localization_site.
target_column = y_raw.columns[0]

# Convert the target DataFrame to a Series.
y_series = y_raw[target_column].astype(str)

# Display the number of observations in each class.
class_counts = y_series.value_counts().sort_values(ascending=False)

print("Number of classes:", y_series.nunique())
display(class_counts.to_frame(name="sample_count"))

# Plot the class distribution.
plt.figure(figsize=(10, 5))
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.title("Class Distribution in the Yeast Dataset")
plt.xlabel("Localization class")
plt.ylabel("Number of samples")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Prepare features and labels

### 4.1 Convert features to numeric values

The Yeast dataset is documented as having eight real-valued input features. We explicitly convert the feature matrix to numeric values and check for invalid entries.

### 4.2 Encode class labels

Neural-network classification models operate on numeric targets. `LabelEncoder` maps the original class names to integer IDs from $0$ to $K-1$.

For categorical cross-entropy, we then convert these integer IDs into one-hot vectors. For example, a class with index 2 among 5 classes becomes:

$$
[0, 0, 1, 0, 0]
$$

The mapping is retained so that predictions can later be translated back to the original class names.

In [ ]:
# Convert all feature columns to numeric values.
X_df = X_raw.apply(pd.to_numeric, errors="coerce")

# Check whether conversion introduced missing values.
if X_df.isna().any().any():
    raise ValueError("Unexpected non-numeric or missing values were found in the features.")

# Convert the feature matrix to float32 for efficient neural-network computation.
X = X_df.to_numpy(dtype=np.float32)

# Encode the categorical labels as integer IDs.
label_encoder = LabelEncoder()
y_integer = label_encoder.fit_transform(y_series)

class_names = label_encoder.classes_
num_classes = len(class_names)
num_features = X.shape[1]

# Convert integer labels to one-hot encoded vectors.
y_one_hot = keras.utils.to_categorical(y_integer, num_classes=num_classes)

print("Feature matrix:", X.shape)
print("Integer labels:", y_integer.shape)
print("One-hot labels:", y_one_hot.shape)
print("Class names:", class_names)

## 5. Split the data

We create three independent subsets:

- **Training set:** used to update model weights.
- **Validation set:** used to monitor training and make model-development decisions.
- **Test set:** reserved for the final evaluation.

We use stratified splitting so that each subset approximately preserves the original class proportions. The test set must not influence preprocessing, architecture selection, or training decisions.

The split below allocates approximately 60% to training, 20% to validation, and 20% to testing.

In [ ]:
# First, reserve 20% of the data for the final test set.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y_integer,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_integer
)

# Split the remaining 80% equally into training and validation sets.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_train_val
)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

### 5.1 Verify class representation in each split

Stratification helps preserve class proportions, but rare classes may still have very few observations in a subset. Inspect the counts before interpreting per-class metrics.

In [ ]:
# Compare class counts across the three subsets.
split_counts = pd.DataFrame({
    "train": pd.Series(y_train).map(dict(enumerate(class_names))).value_counts(),
    "validation": pd.Series(y_val).map(dict(enumerate(class_names))).value_counts(),
    "test": pd.Series(y_test).map(dict(enumerate(class_names))).value_counts()
}).fillna(0).astype(int)

display(split_counts)

## 6. Feature scaling without data leakage

Neural networks are generally easier to optimize when numerical features have comparable scales.

Standardization transforms each feature using:

$$
z = \frac{x-\mu_{\text{train}}}{\sigma_{\text{train}}}
$$

The mean and standard deviation must be estimated **only from the training set**. Applying `fit` to the complete dataset would allow information from the validation and test sets to influence preprocessing, causing data leakage.

In [ ]:
# Create the scaler and fit it only on the training features.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Apply the training-set transformation to validation and test data.
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert to float32 for Keras.
X_train_scaled = X_train_scaled.astype(np.float32)
X_val_scaled = X_val_scaled.astype(np.float32)
X_test_scaled = X_test_scaled.astype(np.float32)

print("Training feature means (approximately zero):")
print(X_train_scaled.mean(axis=0).round(3))

print("\nTraining feature standard deviations (approximately one):")
print(X_train_scaled.std(axis=0).round(3))

## 7. Build a multiclass neural network

A feedforward neural network transforms the input through a sequence of trainable layers.

For a hidden layer:

$$
\mathbf{h} = \phi(\mathbf{W}\mathbf{x}+\mathbf{b})
$$

where $\phi$ is a nonlinear activation function. We use ReLU in the hidden layers.

The output layer has one unit per class. Softmax converts the output logits into a probability distribution:

$$
\hat{p}_k =
\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}
$$

The probabilities sum to one. The predicted class is the index with the largest probability.

### Model architecture

- Input: 8 standardized numerical features
- Hidden layer 1: 64 units, ReLU
- Dropout: 0.25
- Hidden layer 2: 32 units, ReLU
- Output: 10 units, softmax

In [ ]:
# Build a compact multilayer perceptron for multiclass classification.
model = keras.Sequential([
    keras.Input(shape=(num_features,), name="protein_features"),

    layers.Dense(64, activation="relu", name="hidden_1"),
    layers.Dropout(0.25, name="dropout"),

    layers.Dense(32, activation="relu", name="hidden_2"),

    layers.Dense(num_classes, activation="softmax", name="class_probabilities")
], name="yeast_localization_classifier")

model.summary()

## 8. Compile the model

We use:

- **Optimizer:** Adam, an adaptive gradient-based optimizer.
- **Loss:** categorical cross-entropy, suitable for one-hot encoded multiclass labels.
- **Metric:** accuracy, which measures the proportion of correctly classified samples.

For one-hot target vector $\mathbf{y}$ and predicted probabilities $\hat{\mathbf{p}}$, categorical cross-entropy is:

$$
L = -\sum_{k=1}^{K} y_k \log(\hat{p}_k)
$$

Accuracy is useful, but it does not describe how performance is distributed across classes. We will calculate additional metrics after training.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

## 9. Account for class imbalance

The class distribution is not uniform. One option is to assign larger training weights to underrepresented classes.

For class $k$, a common balanced weight is:

$$
w_k = \frac{N}{K\,N_k}
$$

where $N$ is the number of training samples, $K$ is the number of classes, and $N_k$ is the number of training samples in class $k$.

Class weighting changes the contribution of each observation to the training loss. It does not create new observations, and it cannot fully solve the uncertainty associated with extremely rare classes.

In [ ]:
# Compute class weights from the training labels only.
classes_in_train = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes_in_train,
    y=y_train
)

class_weight = {
    int(class_id): float(weight)
    for class_id, weight in zip(classes_in_train, weights)
}

print("Class weights:")
for class_id, weight in class_weight.items():
    print(f"{class_names[class_id]} (ID {class_id}): {weight:.3f}")

## 10. Train the model

We use early stopping to reduce unnecessary training when validation loss stops improving.

The best model weights are restored at the end of training. The validation set is used for this training decision; the test set remains untouched.

In [ ]:
# Stop training if validation loss does not improve for several epochs.
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train_scaled,
    y_train_one_hot if "y_train_one_hot" in globals() else keras.utils.to_categorical(y_train, num_classes),
    validation_data=(
        X_val_scaled,
        keras.utils.to_categorical(y_val, num_classes)
    ),
    epochs=150,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[early_stopping],
    verbose=1
)

## 11. Visualize the learning curves

Learning curves help us inspect optimization behavior and possible overfitting.

- If training and validation loss both remain high, the model may be underfitting.
- If training loss decreases while validation loss increases, overfitting may be occurring.
- A small gap between training and validation curves is not proof of good generalization; final performance must be assessed on the held-out test set.

In [ ]:
# Plot training and validation loss.
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Categorical cross-entropy")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Plot training and validation accuracy.
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Evaluate on the test set

The test set is evaluated only after the model-development decisions are complete.

We report accuracy, macro-averaged precision, recall, and F1-score.

Macro averaging calculates each class metric independently and then takes the unweighted mean. Therefore, each class contributes equally to the macro score, regardless of its sample count.

This is particularly informative when class frequencies are imbalanced.

In [ ]:
# Evaluate the restored best model on the test set.
test_loss, test_accuracy = model.evaluate(
    X_test_scaled,
    keras.utils.to_categorical(y_test, num_classes),
    verbose=0
)

# Obtain predicted class probabilities and class IDs.
y_probability = model.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(y_probability, axis=1)

# Calculate macro-averaged metrics.
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print(f"Test loss:       {test_loss:.4f}")
print(f"Test accuracy:   {test_accuracy:.4f}")
print(f"Macro precision:{macro_precision:.4f}")
print(f"Macro recall:    {macro_recall:.4f}")
print(f"Macro F1-score:  {macro_f1:.4f}")

### 12.1 Compare against a majority-class baseline

A simple baseline predicts the most frequent class for every test sample. A neural network should be compared with a meaningful baseline rather than evaluated in isolation.

In [ ]:
# Predict the most frequent training class for every test observation.
majority_class = np.bincount(y_train).argmax()
baseline_predictions = np.full(shape=len(y_test), fill_value=majority_class)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)

baseline_macro_f1 = precision_recall_fscore_support(
    y_test,
    baseline_predictions,
    average="macro",
    zero_division=0
)[2]

print(f"Majority-class baseline accuracy: {baseline_accuracy:.4f}")
print(f"Majority-class baseline macro F1: {baseline_macro_f1:.4f}")
print(f"Neural network test accuracy:    {test_accuracy:.4f}")
print(f"Neural network macro F1:          {macro_f1:.4f}")

## 13. Classification report

The classification report presents precision, recall, and F1-score for each class.

- **Precision:** among samples predicted as a class, the fraction that truly belongs to that class.
- **Recall:** among samples belonging to a class, the fraction correctly identified.
- **F1-score:** the harmonic mean of precision and recall.
- **Support:** the number of true test observations in that class.

For rare classes, these values can be unstable because only a small number of test observations may be available. Always interpret per-class metrics alongside support.

In [ ]:
report = classification_report(
    y_test,
    y_pred,
    labels=np.arange(num_classes),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T
display(report_df.round(3))

## 14. Confusion matrix

A confusion matrix compares true classes with predicted classes. Rows represent true labels and columns represent predicted labels.

Off-diagonal values show which classes are confused with one another. The raw counts are useful, but a normalized confusion matrix can make patterns easier to compare when class sizes differ.

In [ ]:
# Calculate the confusion matrix using a fixed class order.
cm = confusion_matrix(y_test, y_pred, labels=np.arange(num_classes))

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

In [ ]:
# Normalize each row to show the fraction of each true class assigned to each prediction.
row_sums = cm.sum(axis=1, keepdims=True)
cm_normalized = np.divide(
    cm,
    row_sums,
    out=np.zeros_like(cm, dtype=float),
    where=row_sums != 0
)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0,
    vmax=1,
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("Row-Normalized Confusion Matrix")
plt.tight_layout()
plt.show()

## 15. Inspect individual predictions

The softmax output represents the model's estimated class probabilities. A high maximum probability is not automatically a guarantee that a prediction is correct or well-calibrated.

We inspect a few test observations and compare the predicted class with the true class.

In [ ]:
# Display a small sample of test predictions.
n_examples = min(10, len(y_test))
example_indices = np.arange(n_examples)

prediction_table = pd.DataFrame({
    "true_class": label_encoder.inverse_transform(y_test[example_indices]),
    "predicted_class": label_encoder.inverse_transform(y_pred[example_indices]),
    "maximum_probability": y_probability[example_indices].max(axis=1)
})

prediction_table["correct"] = (
    prediction_table["true_class"] == prediction_table["predicted_class"]
)

display(prediction_table)

## 16. Save and reload the model

The `.keras` format stores the model architecture, weights, and training configuration.

The feature scaler and label encoder are separate preprocessing objects. In a real deployment or reproducible research workflow, they must also be saved and versioned with the model.

In [ ]:
from pathlib import Path
import joblib

# Create an output directory for model artifacts.
output_dir = Path("artifacts")
output_dir.mkdir(exist_ok=True)

# Save the Keras model.
model_path = output_dir / "yeast_localization_classifier.keras"
model.save(model_path)

# Save preprocessing objects separately.
joblib.dump(scaler, output_dir / "feature_scaler.joblib")
joblib.dump(label_encoder, output_dir / "label_encoder.joblib")

print("Saved model and preprocessing artifacts to:", output_dir.resolve())

In [ ]:
# Reload the model and verify that predictions are reproducible.
reloaded_model = keras.models.load_model(model_path)

reloaded_probabilities = reloaded_model.predict(X_test_scaled[:5], verbose=0)

print("Original probabilities (first sample):")
print(y_probability[0].round(5))

print("\nReloaded probabilities (first sample):")
print(reloaded_probabilities[0].round(5))

print(
    "\nMaximum absolute difference:",
    np.max(np.abs(y_probability[:5] - reloaded_probabilities))
)

## 17. Interpretation and scientific limitations

Before drawing conclusions, consider the following limitations:

1. **Small sample size:** 1,484 observations are limited for training a neural network with many parameters.
2. **Class imbalance:** some localization classes are much rarer than others. Their test metrics may be highly uncertain.
3. **Feature representation:** the model uses the eight supplied numerical features. It does not learn directly from protein sequences or experimental images.
4. **Benchmark context:** performance on this dataset does not establish generalization to other organisms, laboratories, or experimental settings.
5. **Model comparison:** a neural network should be compared with simpler baselines such as logistic regression, decision trees, random forests, and support vector machines.
6. **Biological interpretation:** predictive associations are not evidence of biological causation or experimental confirmation.

A rigorous study would also report the split strategy, random seed, software versions, class-wise metrics, and uncertainty across repeated runs or cross-validation folds.

## 18. Exercises

### Exercise 1 — Architecture
Train a smaller model with one hidden layer of 16 units. Compare its test macro F1-score with the current model.

### Exercise 2 — Learning rate
Repeat training with learning rates of `1e-2`, `1e-3`, and `1e-4`. Compare convergence and validation loss.

### Exercise 3 — Class weighting
Train the model without `class_weight`. Compare class-wise recall and macro F1-score with the weighted model.

### Exercise 4 — Metrics
Explain why accuracy and macro F1-score can lead to different conclusions on an imbalanced dataset.

### Exercise 5 — Error analysis
Identify the classes with the lowest recall. How many test samples support each estimate? What additional data would improve confidence?

### Exercise 6 — Baseline comparison
Train a multinomial logistic regression model using the same train/validation/test split and training-only scaling. Compare its performance with the neural network.

### Research challenge
Design a reproducible comparison of at least three classifiers. Keep the test set untouched until all model and hyperparameter decisions have been made. Report macro F1, balanced accuracy, class-wise recall, and a confusion matrix.

## 19. Key takeaways

- Multiclass classification predicts one label from more than two possible classes.
- A softmax output layer produces a probability distribution over classes.
- Categorical cross-entropy is a standard loss for one-hot encoded multiclass targets.
- Stratified splitting helps preserve class proportions across data subsets.
- Preprocessing parameters must be learned from the training data only.
- Class imbalance makes accuracy alone insufficient.
- Macro-averaged metrics and class-wise reports provide a more complete evaluation.
- A neural network should be compared with simple baselines, especially on small tabular datasets.

## 20. References

1. Nakai, K. (1991). **Yeast** [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5KG68
2. Chollet, F. (2021). *Deep Learning with Python* (2nd ed.). Manning Publications.
3. Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly Media.
4. Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press.
5. Keras documentation: https://keras.io/

**Dataset license:** Creative Commons Attribution 4.0 International (CC BY 4.0). Cite the UCI dataset and its DOI when reusing it.